# Geko Fitting Demo: New Code Structure

This notebook mirrors `simple_fit_demo.ipynb` but explains the restructured architecture in detail.
The run call itself is identical — what changes is how the code is organised internally,
and how you control it through `FitConfiguration`.

---

## High-level architecture

```
FitConfiguration
  │
  │  morphology_model = 'Sersic'          ← chosen from MORPH_REGISTRY
  │  rotation_components = ['Arctan']     ← chosen from COMPONENT_REGISTRY
  │  morph_prior_overrides = {...}
  │  geom_prior_overrides  = {...}
  │  rot_prior_overrides   = {...}
  │  fixed_params          = {...}
  │
  └─► fitting.py assembles:
        │
        ├── SersicMorphology()          ← MorphologyModel
        │     parameters: amplitude, r_eff, n, PA_morph, xc_morph, yc_morph
        │     priors: defaults → PySersic → config overrides
        │
        ├── SHARED_KINEMATIC_SPEC       ← list of ParameterSpec
        │     parameters: PA, i, sigma0, x0_vel, y0_vel, v0
        │     priors: defaults → config geom_prior_overrides
        │
        └── CompositeRotationCurve([ArctanComponent()])
              parameters: Va, r_t
              v(r) = sqrt( Va² · (2/π · arctan(r/r_t))² )
              priors: defaults → config rot_prior_overrides
```

These three objects live inside `GalaxyModel`, which itself lives inside `GrismFitter`
(the top-level object that was previously called `DiskModel`).

```
GrismFitter
  └── GalaxyModel
        ├── morph_model  : SersicMorphology
        ├── shared_kin_specs : [PA, i, sigma0, x0_vel, y0_vel, v0]
        └── rot_model    : CompositeRotationCurve
              └── components: [ArctanComponent]   (or [SersicComponent, NFWComponent], etc.)
```

## How parameters are defined: `ParameterSpec`

Every fitted parameter is described by a `ParameterSpec` object. This is the single source
of truth for a parameter's name, prior type, bounds, and whether it is fixed.

```python
@dataclass
class ParameterSpec:
    name:        str          # e.g. 'Va'
    label:       str          # LaTeX label for axis labels, e.g. r'$V_a$ [km/s]'
    title:       str          # Short LaTeX title for corner plot, e.g. r'$V_a$'
    prior_type:  str          # 'Uniform' | 'Normal' | 'TruncatedNormal'
    prior_min:   float        # lower bound (Uniform / TruncatedNormal)
    prior_max:   float        # upper bound (Uniform / TruncatedNormal)
    prior_mu:    float        # centre (Normal / TruncatedNormal)
    prior_std:   float        # width  (Normal / TruncatedNormal)
    fixed:       bool         # if True, not sampled
    fixed_value: float | str  # float → pin to value; str → link to another parameter
```

When `sample_specs()` is called during MCMC, it reads these specs and calls the appropriate
numpyro distribution. Fixed parameters are registered as `numpyro.deterministic` sites
so they still appear in the posterior trace (useful for diagnostics), but they don't
consume any degrees of freedom.

The reparameterisation trick is used throughout: MCMC samples an `unscaled_X ~ Uniform(0,1)`
or `unscaled_X ~ Normal(0,1)`, then `X` is computed as a deterministic transformation.
This is why you see `unscaled_Va` in the NUTS diagnostics table but `Va` in the posterior.

## The model registries

Models are looked up by string name from two registries defined in `geko/morph_models.py`
and `geko/rotation_models.py`.

```python
# geko/morph_models.py
MORPH_REGISTRY = {
    'Sersic': SersicMorphology,
}

# geko/rotation_models.py
COMPONENT_REGISTRY = {
    'Arctan': ArctanComponent,
    'Sersic': SersicComponent,   # needs kpc_per_px (physical scale)
    'NFW':    NFWComponent,      # needs kpc_per_px
}
```

When you set `rotation_components = ['Arctan']` in `FitConfiguration`, `fitting.py`
does exactly:

```python
components = []
for name in config.rotation_components:
    cls = COMPONENT_REGISTRY[name]
    comp = cls()
    if cls.NEEDS_PHYSICAL_SCALE:
        comp.kpc_per_px = kpc_per_px   # computed from redshift + pixel scale
    components.append(comp)
rot_model = CompositeRotationCurve(components)
```

The physical scale (`kpc_per_px`) is only needed for mass-based components (Sersic, NFW)
that convert pixel radii to physical radii. The arctan model is purely phenomenological
and works in pixel units.

## How the rotation curve is composed

`CompositeRotationCurve` holds a list of components. Each component implements `v_sq(r, all_params)`
— its contribution to `v²(r)`. The total line-of-sight velocity is:

```
v_circ(r) = sqrt( v²_comp1(r) + v²_comp2(r) + ... )
v_los     = v_circ(r) · sin(i) · (y_rot / r)
```

where `y_rot` is the sky-plane coordinate along the projected major axis after deprojection.

This means:
- **One component** (Arctan): `v(r) = Va · (2/π) · arctan(r / r_t)` — phenomenological flat curve
- **Two components** (Sersic + NFW): `v(r) = sqrt(v²_stellar(r) + v²_halo(r))` — physically motivated
- **Three components** (Sersic + NFW + gas disc): trivially extensible

The inference code (`inference_model_parametric_multi` in `models.py`) calls:

```python
velocities = galaxy_model.velocity_field(X_grid, Y_grid, PA, i, all_params)
```

where `all_params` is the merged dict of morphology + geometry + rotation parameters.
This single call works regardless of how many components are in the rotation curve.

## How priors are set: priority order

For morphology parameters, three sources of prior information are applied in sequence —
later sources override earlier ones:

```
1. Defaults (hardcoded in each ParameterSpec inside SersicMorphology)
       ↓  (overwritten by)
2. PySersic catalog  — if a .cat file is provided
       ↓  (overwritten by)
3. morph_prior_overrides in FitConfiguration  — always wins
```

For geometry and rotation parameters:
```
1. Defaults (hardcoded in SHARED_KINEMATIC_SPEC / ArctanComponent._DEFAULT_PARAMETERS)
       ↓  (overwritten by)
2. geom_prior_overrides / rot_prior_overrides in FitConfiguration
```

This means you can always be confident that whatever you put in `FitConfiguration`
takes effect, regardless of what PySersic returned.

In [ ]:
from geko.fitting import run_geko_fit
from geko.config import FitConfiguration, MCMCSettings

import jax
import numpyro
jax.config.update('jax_enable_x64', True)
numpyro.set_host_device_count(2)

print('JAX version:', jax.__version__)
print('Devices:', jax.devices())

## Step 1: Data parameters

These are unchanged from the original demo — same files, same source.

### The `field` parameter

`field` tells geko which JWST survey programme your data comes from. Predefined options
automatically set the grism file naming convention, the PSF, and the rotation angle
between the imaging and grism frames:

| `field` | Programme | Grism file pattern | PSF |`theta_rot` |
|---|---|---|---|---|
| `'GOODS-S-FRESCO'` | [FRESCO](https://www.stsci.edu/jwst/phase2-public/1895.pdf) (PID 1895), GOODS-S | `spec_2d_FRESCO_{filter}_ID{id}_comb.fits` | `mpsf_jw018950.gs.f444w.fits` | 0° |
| `'GOODS-N'` | FRESCO (PID 1895), GOODS-N | `spec_2d_GDN_{filter}_ID{id}_comb.fits` | `mpsf_jw018950.gn.f444w.fits` | 230.5° |
| `'GOODS-N-CONGRESS'` | [CONGRESS](https://www.stsci.edu/jwst/phase2-public/3577.pdf) (PID 3577), GOODS-N | `spec_2d_GDN_{filter}_ID{id}_comb.fits` | `mpsf_jw035770.f356w.fits` | 228.2° |
| `'manual'` | Any | specified by `manual_grism_file` | specified by `manual_psf_name` | specified by `manual_theta_rot` |

For any other survey or custom data, use `field='manual'` and supply the three extra
parameters:
- `manual_grism_file`: FITS filename, looked up as `<save_runs_path>/<output>/<filename>`
- `manual_psf_name`: PSF FITS filename, looked up as `<save_runs_path>/psfs/<filename>`
- `manual_theta_rot`: rotation angle in degrees (see below)

### What is `theta_rot`?

`theta_rot` is the rotation angle (degrees) between the grism observation and the reference
imaging frame used for morphology.

geko models galaxy morphology from a direct NIRCam image (via PySersic), then projects
that morphology into the grism dispersion frame to predict the 2D spectrum. If the grism
and imaging were observed at different position angles on the sky, the morphological PA
must be rotated accordingly before the projection. `theta_rot` encodes that relative
orientation.

For the predefined fields the angle is hardcoded because the relative orientation between
the JADES NIRCam imaging and each grism survey is fixed. For `manual` data, compute it
from your FITS headers:

```
theta_rot = PA_grism_observation - PA_imaging_observation   (mod 360°)
```

where `PA` is typically the `PA_V3` or `ROLL_REF` keyword in each FITS primary header.

In [ ]:
geko_path = '/Users/lola/geko/'   # change to your path

source_id            = 191250
field                = 'manual'
output_name          = 'my_galaxy'
master_catalog       = geko_path + 'demo/simple_fit_demo_files/catalogs/my_galaxies_cat'
emission_line        = 'H_alpha'   # emission line name — used to look up the line in the master catalog
parametric           = True
save_runs_path       = geko_path + 'demo/simple_fit_demo_files/'

manual_psf_name      = 'webbPSF_F444W.fits'
manual_theta_rot     = 0.0
manual_pysersic_file = 'summary_191250_image_F150W_svi.cat'
manual_grism_file    = 'spec_2d_FRESCO_F444W_ID191250_comb.fits'

grism_filter         = 'F444W'
delta_wave_cutoff    = 0.02
factor               = 1     # set low for demo speed (use 5 in real runs)
wave_factor          = 1     # set low for demo speed (use 9 in real runs)

num_chains  = 1
num_warmup  = 100
num_samples = 100

## Step 2: Inspect the default parameter specs

Before building a config, it's useful to see what parameters each model has by default
and what their default prior ranges are. This tells you exactly what keys to use
in the override dicts.

In [ ]:
from geko.morph_models import SersicMorphology, MORPH_REGISTRY
from geko.rotation_models import ArctanComponent, CompositeRotationCurve, COMPONENT_REGISTRY
from geko.param_spec import SHARED_KINEMATIC_SPEC, all_param_specs
from copy import deepcopy

morph = SersicMorphology()
rot   = CompositeRotationCurve([ArctanComponent()])
shared = deepcopy(SHARED_KINEMATIC_SPEC)

print('Available morphology models: ', list(MORPH_REGISTRY.keys()))
print('Available rotation components:', list(COMPONENT_REGISTRY.keys()))
print()

In [ ]:
# Print default priors for all parameters
# This is exactly what geko uses if you pass config=None

print(f'  {"Parameter":<15} {"Group":<12} {"Prior type":<18} {"Bounds / mu±std"}')
print('  ' + '-'*65)

all_specs = all_param_specs(morph, shared, rot)
for spec in all_specs:
    if spec.prior_type == 'Uniform':
        bounds = f'[{spec.prior_min}, {spec.prior_max if spec.prior_max is not None else "r_eff"}]'
    elif spec.prior_type in ('Normal', 'TruncatedNormal'):
        bounds = f'mu={spec.prior_mu}, std={spec.prior_std}'
        if spec.prior_min is not None:
            bounds += f', [{spec.prior_min}, {spec.prior_max}]'
    else:
        bounds = ''

    if spec in morph.parameters:
        group = 'morphology'
    elif spec in rot.parameters:
        group = 'rotation'
    else:
        group = 'geometry'

    print(f'  {spec.name:<15} {group:<12} {spec.prior_type:<18} {bounds}')

## Step 3: Build a `FitConfiguration`

### Choosing a morphology model

Set `morphology_model` to any key in `MORPH_REGISTRY`. Currently only `'Sersic'` is implemented.

```python
FitConfiguration(morphology_model='Sersic')   # default
```

When a new morphology model is added (e.g. a PSF-convolved exponential disc), it will appear
in `MORPH_REGISTRY` and you switch to it here — no other code changes needed.

### Choosing rotation curve components

Set `rotation_components` to any list of keys from `COMPONENT_REGISTRY`.
Components are **additive in v²**: the total circular velocity at radius `r` is
`v_circ(r) = sqrt( Σ v²_i(r) )`.

```python
# Phenomenological (default):
rotation_components = ['Arctan']
    # Parameters: Va (asymptotic velocity), r_t (turnover radius)

# Physical — stellar disc only:
rotation_components = ['Sersic']
    # Parameters: log_M_star (log stellar mass)
    # Reads r_eff and n from the morphology model

# Physical — stellar disc + dark matter halo:
rotation_components = ['Sersic', 'NFW']
    # Parameters: log_M_star, log_M_halo, c_halo
    # Sersic and NFW components require kpc_per_px (from redshift)
```

The override keys for rotation parameters follow the same `{name}_{min|max|mu|std}` convention:

| Component | Parameters | Example override key |
|---|---|---|
| `Arctan` | `Va`, `r_t` | `Va_max`, `r_t_mu` |
| `Sersic` | `log_M_star` | `log_M_star_min` |
| `NFW` | `log_M_halo`, `c_halo` | `log_M_halo_max`, `c_halo_max` |

### Option A — defaults only (PySersic morphology, default priors)

Passing `config=None` to `run_geko_fit` is equivalent to this. PySersic priors
are loaded automatically and used for all morphology parameters.

In [ ]:
fit_config = FitConfiguration(
    mcmc=MCMCSettings(
        num_chains=num_chains,
        num_warmup=num_warmup,
        num_samples=num_samples,
    )
    # morphology_model='Sersic' is the default
    # rotation_components=['Arctan'] is the default
    # all override dicts default to {}
)
fit_config.print_summary()

### Option B — override specific priors

Each override dict is sparse — only set what you want to change.

The key convention is `{param_name}_{min|max|mu|std}`:
- `_mu` and `_std` set the centre and width of a Normal or TruncatedNormal prior
- `_min` and `_max` set the bounds of a Uniform or TruncatedNormal prior

**Which dict for which parameter?**

| Dict | Parameters |
|---|---|
| `morph_prior_overrides` | `amplitude`, `r_eff`, `n`, `PA_morph`, `xc_morph`, `yc_morph` |
| `geom_prior_overrides` | `PA`, `i`, `sigma0`, `x0_vel`, `y0_vel`, `v0` |
| `rot_prior_overrides` | `Va`, `r_t` (Arctan) — or `log_M_star`, `log_M_halo`, `c_halo` for mass models |

In [ ]:
fit_config = FitConfiguration(
    mcmc=MCMCSettings(
        num_chains=num_chains,
        num_warmup=num_warmup,
        num_samples=num_samples,
    ),
    morphology_model='Sersic',
    rotation_components=['Arctan'],

    # Morphology: only expand the r_eff upper bound beyond the PySersic default
    morph_prior_overrides={
        'r_eff_max': 12.0,
    },

    # Geometry / shared kinematics
    geom_prior_overrides={
        'i_min':      20.0,    # avoid very face-on
        'i_max':      80.0,    # avoid edge-on
        'sigma0_min': 10.0,
        'sigma0_max': 200.0,
    },

    # Rotation curve (Arctan component)
    rot_prior_overrides={
        'Va_min': 0.0,
        'Va_max': 400.0,
    },
)
fit_config.print_summary()

### Option C — fix or link parameters

`fixed_params` lets you remove a parameter from the MCMC entirely.

- **Float value** → pins the parameter to that number. It still appears in the
  posterior trace as a constant (via `numpyro.deterministic`), but is absent from
  the corner plot.
- **String value** → links the parameter to another already-sampled parameter.
  Useful when you want morphological and kinematic PA to be the same.

```python
fixed_params = {
    'i': 60.0,           # pin inclination to 60°
    'PA_morph': 'PA',    # link morphological PA to kinematic PA
}
```

Linked parameters must refer to a parameter that is sampled earlier in the sequence.
The order is: morphology → geometry → rotation.

In [ ]:
config_with_fixed = FitConfiguration(
    mcmc=MCMCSettings(num_chains=num_chains, num_warmup=num_warmup, num_samples=num_samples),
    fixed_params={
        'i': 60.0,
        # 'PA_morph': 'PA',   # uncomment to link PA_morph → PA
    }
)
config_with_fixed.print_summary()

### Option D — switching to a physical rotation model (future)

Once `SersicComponent` and `NFWComponent` are implemented, switching looks like this.
The mass-based components need a physical scale (`kpc_per_px`), which `fitting.py`
computes automatically from the source redshift and NIRCam pixel scale.

```python
fit_config = FitConfiguration(
    rotation_components=['Sersic', 'NFW'],
    rot_prior_overrides={
        'log_M_star_min': 9.0,
        'log_M_star_max': 12.0,
        'log_M_halo_min': 10.0,
        'log_M_halo_max': 14.0,
        'c_halo_min':  1.0,
        'c_halo_max': 30.0,
    },
    mcmc=MCMCSettings(...),
)
```

The `SersicComponent` reads `r_eff` and `n` from the morphology posterior
(they are in `all_params` automatically), so the morphology and mass model
share those parameters by default. To decouple them — fitting an independent
mass morphology — use `fixed_params`:

```python
fixed_params = {
    'n_mass':     'n',       # link mass Sersic index to light Sersic index
    'r_eff_mass': 'r_eff',   # link mass effective radius to light effective radius
}
```

## Step 4: What `run_geko_fit` does internally

This is the sequence inside `fitting.py` when you call `run_geko_fit(..., config=fit_config)`:

```
1. Load and preprocess the grism data
      → GrismObservation object with 2D spectrum, error map, PSF, dispersion table

2. Build the morphology model
      morph_model = MORPH_REGISTRY['Sersic']()

3. Build the rotation curve
      components = [COMPONENT_REGISTRY['Arctan']()]
      rot_model  = CompositeRotationCurve(components)

4. Assign models to GrismFitter
      kin_model.galaxy_model.morph_model = morph_model
      kin_model.galaxy_model.rot_model   = rot_model

5. Apply priors — in order:
      a) Defaults already in each ParameterSpec
      b) PySersic catalog (if available) → morph_model.set_priors_from_pysersic()
      c) Config overrides → morph_model.apply_prior_overrides(cfg.morph_prior_overrides)
                            apply_overrides(shared_kin_specs, cfg.geom_prior_overrides)
                            rot_comp.apply_prior_overrides(cfg.rot_prior_overrides)

6. Apply fixed parameters
      kin_model.galaxy_model.apply_fixed_params(cfg.fixed_params)

7. Run MCMC
      numpyro NUTS with the assembled model

8. Post-process
      compute_posterior_means_parametric()  → morph_means, rot_means dicts
      compute_parametrix_flux_posterior()   → flux map at posterior mean
      compute_model_parametric()            → model grism map at posterior mean

9. Save results and generate plots
      postprocess.write_results_table()     → {source_id}_results
      plotting.plot_disk_summary()          → {source_id}_summary.png
```

## Step 5: Run the fit

In [ ]:
inference_data = run_geko_fit(
    output=output_name,
    master_cat=master_catalog,
    line=emission_line,
    parametric=parametric,
    save_runs_path=save_runs_path,
    num_chains=num_chains,
    num_warmup=num_warmup,
    num_samples=num_samples,
    source_id=source_id,
    field=field,
    grism_filter=grism_filter,
    delta_wave_cutoff=delta_wave_cutoff,
    factor=factor,
    wave_factor=wave_factor,
    config=fit_config,
    manual_psf_name=manual_psf_name,
    manual_theta_rot=manual_theta_rot,
    manual_pysersic_file=manual_pysersic_file,
    manual_grism_file=manual_grism_file,
)

## Step 6: Inspect the posterior results

### Results table

The results table is written by `postprocess.py`. Column names are derived dynamically
from `all_param_specs()` — so if you switch rotation models, new columns appear
automatically (`log_M_star_50`, `log_M_halo_50`, etc.) with no code changes.

In [ ]:
import os
import arviz as az
from astropy.table import Table

output_dir = os.path.join(save_runs_path, output_name)
fit_results = Table.read(os.path.join(output_dir, f'{source_id}_results'), format='ascii')
print(fit_results)

### The result dicts on `GrismFitter`

After `compute_model_parametric()` is called (automatically inside `run_geko_fit`),
results are stored in structured dicts on the `GrismFitter` object:

```python
kin_model.morph_means        # dict  {param_name: posterior_median}
kin_model.morph_quantiles    # dict  {param_name: {'16': q16, '84': q84}}
kin_model.rot_means          # dict  {param_name: posterior_median}
kin_model.rot_quantiles      # dict  {param_name: {'16': q16, '84': q84}}
```

For convenience, backward-compatible named attributes are also set via `setattr`,
so code that already reads `kin_model.Va_mean` or `kin_model.r_eff_mean` continues
to work unchanged.

In [ ]:
# Show what the parameter spec list looks like — this is what drives the
# corner plot, results table columns, and MCMC var_names
from geko.param_spec import all_param_specs
from copy import deepcopy

morph  = SersicMorphology()
rot    = CompositeRotationCurve([ArctanComponent()])
shared = deepcopy(SHARED_KINEMATIC_SPEC)

specs = all_param_specs(morph, shared, rot)

print('Parameters included in corner plot and results table:')
for s in specs:
    status = 'fixed' if s.fixed else 'sampled'
    print(f'  {s.name:<18} {status}   label: {s.label}')

### Reading the posterior directly from arviz

The `inference_data` object returned by `run_geko_fit` is a standard
`arviz.InferenceData`. Each fitted parameter appears as a variable in
`inference_data.posterior`. Fixed parameters also appear (as constants).

The reparameterised `unscaled_X` variables are also present — these are what
NUTS actually sampled. The `X` variables are the physically meaningful
deterministic transformations.

In [ ]:
# List all variables in the posterior (excluding unscaled_ reparameterisations)
posterior_vars = [v for v in inference_data.posterior.data_vars if not v.startswith('unscaled_')]
print('Posterior variables:')
for v in sorted(posterior_vars):
    med = float(inference_data.posterior[v].median())
    print(f'  {v:<20s}: median = {med:.4f}')

## Step 7: Output files

Output files are identical to the original demo:

| File | Contents |
|---|---|
| `{source_id}_output` | Full MCMC posterior (arviz InferenceData, NetCDF) |
| `{source_id}_results` | Median ± 1σ for all parameters + derived quantities (v_re, v/σ) |
| `{source_id}_summary.png` | Observed / model / residuals + 1D velocity and dispersion profiles |
| `{source_id}_v_sigma_corner.png` | v/σ posterior |
| `{source_id}_summary_corner.png` | Full parameter corner plot |

The corner plot and results table are now fully dynamic — their columns and axes are
derived from `all_param_specs(morph_model, shared_kin_specs, rot_model)` at plot time.
Fixed parameters are automatically excluded from the corner plot.

In [ ]:
from IPython.display import Image, display

summary_plot = os.path.join(output_dir, f'{source_id}_summary.png')
if os.path.exists(summary_plot):
    display(Image(filename=summary_plot, width=800))
else:
    print(f'Plot not found: {summary_plot}')

In [ ]:
corner_plot = os.path.join(output_dir, f'{source_id}_summary_corner.png')
if os.path.exists(corner_plot):
    display(Image(filename=corner_plot, width=800))
else:
    print(f'Plot not found: {corner_plot}')

## Step 8: MCMC diagnostics

In [ ]:
import matplotlib.pyplot as plt

az.plot_trace(inference_data, var_names=['Va', 'sigma0', 'PA', 'i'])
plt.tight_layout()
plt.show()

## Quick-reference: old API → new API

| Old | New |
|---|---|
| `DiskModel` | `GrismFitter` |
| `MorphologyPriors(PA_mean=x, inc_mean=y)` | `geom_prior_overrides={'PA_mu': x, 'i_mu': y}` |
| `KinematicPriors(Va_max=x)` | `rot_prior_overrides={'Va_max': x}` |
| `FitConfiguration(redshift=z, line=l, morphology=..., kinematics=...)` | `FitConfiguration(morph_prior_overrides={}, geom_prior_overrides={}, rot_prior_overrides={})` |
| `kin_model.Va_mean`, `kin_model.r_t_mean` | `kin_model.rot_means['Va']`, `kin_model.rot_means['r_t']` (named attrs still work) |
| `kin_model.r_eff_mean`, `kin_model.n_mean` | `kin_model.morph_means['r_eff']`, `kin_model.morph_means['n']` (named attrs still work) |
| Corner plot var_names hardcoded | Derived dynamically from `all_param_specs(morph, shared, rot)` |
| Adding NFW: edit `models.py`, `fitting.py`, `postprocess.py`, `plotting.py` | Adding NFW: one line in `FitConfiguration(rotation_components=['Sersic','NFW'])` |